In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from model import ScalingLaw, SampleAlpha
from model_general import ScalingLaw as ScalingLawGeneral
from constants import lower_bounds, test_models, delete_models, Y_names_tidy, Y_names, B, lrs, scheduler_factors, reps, n_epochs, random_seed
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed

K = 4
eps = 1e-3
Y_names = Y_names[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Data

In [2]:
# Loading
df = pd.merge(pd.read_csv('data/df_full_v1.csv').drop('Unnamed: 0', axis=1),
              pd.read_csv('data/df_full_v2.csv').drop('Unnamed: 0', axis=1), 
              on=['model', 'family', 'size', 'tokens', 'flops'], how='outer')

# Creating data objects
Y = np.array(df.loc[:,Y_names])
Y = np.clip(Y, a_min=eps, a_max=1-eps)
        
X = np.log(np.array(df.loc[:,['size','tokens']]))
X = np.hstack((X,(X[:,0]*X[:,1])[:,None]))

D = np.array(pd.get_dummies(df.family)).astype(int)
I = np.ones(shape=(D.shape[0],1))
C = np.array([lower_bounds[s] for s in Y_names]).reshape((1,-1))

Training models

In [3]:
test_models.keys()

dict_keys(['meta-llama-3', 'qwen2', 'yi-1.5', 'olmo', 'smollm', 'gemma2'])

In [ ]:
reps = 2
pred_models = {}

for fam in tqdm(test_models.keys()):
    train_idx = [m not in delete_models[fam] for m in df.model]
    test_idx = [m in test_models[fam] for m in df.model]

    mu = np.mean(X[train_idx], axis=0, keepdims=True)
    std = np.std(X[train_idx], axis=0, keepdims=True)

    X_train = (X[train_idx]-mu)/std
    Y_train = Y[train_idx]
    D_train = D[train_idx]
    
    pred_models[fam] = {}
    pred_models[fam]['mu'] = mu
    pred_models[fam]['std'] = std
    pred_models[fam]['X_test'] = X[test_idx]
    pred_models[fam]['Y_test'] = Y[test_idx]

    # Model 1
    pred_models[fam]['model'] = ScalingLaw(K)
    pred_models[fam]['model'].fit(X = X_train,
                                  Y = Y_train,
                                  D = D_train,
                                  C = C,
                                  B = B,
                                  lrs = lrs,
                                  scheduler_factors = scheduler_factors,
                                  reps = reps,
                                  n_epochs = n_epochs,    
                                  verbose = False,
                                  device = device)

    # Model 2
    pred_models[fam]['model-general'] = ScalingLawGeneral(K)
    pred_models[fam]['model-general'].fit(X_mean = X_train,
                                          Y = Y_train,
                                          D = D_train,
                                          C = C,
                                          X_phi = X_train,
                                          B = B,
                                          lrs = lrs,
                                          scheduler_factors = scheduler_factors,
                                          reps = reps,
                                          n_epochs = n_epochs,    
                                          verbose = False,
                                          device = device)
        
    np.save(f"models/prediction_intervals/models_data.npy", pred_models)

  0%|          | 0/6 [00:00<?, ?it/s]

Different lrs:   0%|          | 0/3 [00:00<?, ?it/s]

Different scheduler factors:   0%|          | 0/2 [00:00<?, ?it/s]

Reps:   0%|          | 0/2 [00:00<?, ?it/s]

final grad norm: 0.010142600163817406
final grad norm: 0.005180831532925367


...

Different scheduler factors:   0%|          | 0/2 [00:00<?, ?it/s]

Reps:   0%|          | 0/2 [00:00<?, ?it/s]

final grad norm: 0.028204219415783882
final grad norm: 0.02377721481025219


Reps:   0%|          | 0/2 [00:00<?, ?it/s]

final grad norm: 0.11022424697875977


In [6]:
12*6*3*2*2/60 #min, fam, lr, sch, rep

14.4

In [5]:
13*6*3*2*2 #min, fam, lr, sch, rep

936

Results

In [14]:
df.loc[['meta-llama-3' in m for m in df.family]]

,model,family,size,tokens,flops,arc,hellaswag,mmlu,truthfulqa,winogrande,gsm8k,ifeval,bbh,math,mmlu-pro,gpqa,musr
140,meta-llama-3-70b,meta-llama-3,70.0,15.0,6300.0,0.687700,0.879805,0.792329,0.455624,0.853197,0.768764,0.160000,0.65000,0.170000,0.470000,0.400000,0.450000
141,meta-llama-3-70b-instruct,meta-llama-3-instruct,70.0,15.0,6300.0,0.714200,0.856900,0.800600,0.618100,0.828700,0.854400,0.810000,0.65000,0.230000,0.520000,0.290000,0.420000
142,meta-llama-3-8b,meta-llama-3,8.0,15.0,720.0,0.602400,0.820155,0.664950,0.439523,0.771113,0.453374,0.150000,0.46000,0.030000,0.320000,0.310000,0.360000
143,meta-llama-3-8b-instruct,meta-llama-3-instruct,8.0,15.0,720.0,0.607500,0.785500,0.670700,0.516500,0.745100,0.686900,0.740000,0.50000,0.090000,0.370000,0.260000,0.360000
144,meta-llama-3-8bee,meta-llama-3-ee,8.0,15.0,720.0,0.586177,0.814778,0.660130,0.428964,0.771113,0.398787,0.195066,0.24199,0.041541,0.246639,0.085011,0.062424


In [16]:
pred_models['meta-llama-3']['model'].phi

array([[0.3099017 , 2.2489772 , 1.6521044 , 1.0611217 , 1.6556618 ,
        1.5429889 , 1.2337911 , 1.1897241 , 0.7999948 , 0.99493235,
        1.1130171 , 1.4484007 ]], dtype=float32)

In [9]:
pred_models['meta-llama-3']['df_fam']

,model,family,size,tokens,flops,arc,hellaswag,mmlu,truthfulqa,winogrande,gsm8k,ifeval,bbh,math,mmlu-pro,gpqa,musr
140,meta-llama-3-70b,meta-llama-3,70.0,15.0,6300.0,0.6877,0.879805,0.792329,0.455624,0.853197,0.768764,0.16,0.65,0.17,0.47,0.40,0.45
142,meta-llama-3-8b,meta-llama-3,8.0,15.0,720.0,0.6024,0.820155,0.664950,0.439523,0.771113,0.453374,0.15,0.46,0.03,0.32,0.31,0.36


In [11]:
df.iloc[140:145]

,model,family,size,tokens,flops,arc,hellaswag,mmlu,truthfulqa,winogrande,gsm8k,ifeval,bbh,math,mmlu-pro,gpqa,musr
140,meta-llama-3-70b,meta-llama-3,70.0,15.0,6300.0,0.687700,0.879805,0.792329,0.455624,0.853197,0.768764,0.160000,0.65000,0.170000,0.470000,0.400000,0.450000
141,meta-llama-3-70b-instruct,meta-llama-3-instruct,70.0,15.0,6300.0,0.714200,0.856900,0.800600,0.618100,0.828700,0.854400,0.810000,0.65000,0.230000,0.520000,0.290000,0.420000
142,meta-llama-3-8b,meta-llama-3,8.0,15.0,720.0,0.602400,0.820155,0.664950,0.439523,0.771113,0.453374,0.150000,0.46000,0.030000,0.320000,0.310000,0.360000
143,meta-llama-3-8b-instruct,meta-llama-3-instruct,8.0,15.0,720.0,0.607500,0.785500,0.670700,0.516500,0.745100,0.686900,0.740000,0.50000,0.090000,0.370000,0.260000,0.360000
144,meta-llama-3-8bee,meta-llama-3-ee,8.0,15.0,720.0,0.586177,0.814778,0.660130,0.428964,0.771113,0.398787,0.195066,0.24199,0.041541,0.246639,0.085011,0.062424
